# NLU Prompt to JSON Converter

Notebook chay tren **Google Colab** (GPU runtime), expose Flask API qua **pinggy** tunnel.

Backend (`nluService.ts`) goi: `POST {COLAB_NLU_URL}/api/nlu/parse`  
- Input : `{ "prompt": "..." }`  
- Output: `{ "slots": {...}, "missingSlots": [...], "confidence": 0.0-1.0 }`

### Yeu cau
- Runtime: **GPU** (T4 tren Colab Free)
- Khong can API key - model load tu HuggingFace
- Pinggy tao tunnel mien phi, khong can dang ky

**Chay tuan tu tung cell, hoac Runtime -> Run all.**

In [ ]:
# 1. Cai dependencies
!pip install flask flask-cors transformers accelerate -q
!pip install torch --index-url https://download.pytorch.org/whl/cu118 -q

In [ ]:
# 2. Load Qwen2.5-3B-Instruct tu HuggingFace
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
FLASK_PORT  = 5000

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading model (lan dau ~2 phut)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'Model san sang tren: {next(model.parameters()).device}')

In [ ]:
# 3. NLU Parser - nhan prompt, tra ve {slots, missingSlots, confidence}
# Khop voi contract NluParseResponse trong nluService.ts
import json, re
from datetime import date, datetime, timedelta
# Stop-words/verbs khong nen xuat hien trong experienceKeywords
_EXP_STOPWORDS = {
    'an', 'di', 'xem', 'thich', 'muon', 'duoc', 'co', 'la', 'va', 'voi',
    'choi', 'tham', 'ghe', 'toi', 'minh', 'cung', 'nua', 'nhung',
}
_SYSTEM = (
    'You are a powerful NLU system for a Vietnamese travel planning app.\n'
    'Extract structured information from the user prompt and return ONLY a valid JSON object.\n'
    'No explanation, no markdown — raw JSON only.\n\n'
    'JSON schema:\n'
    '{\n'
    '  "slots": {\n'
    '    "destinationCity":      string or null,\n'
    '    "durationDays":         integer or null,\n'
    '    "startDate":            "YYYY-MM-DD" or null,\n'
    '    "preferredTagNames":    string[],\n'
    '    "experienceKeywords":   string[],\n'
    '    "vibe":                 string[],\n'
    '    "amenities":            string[],\n'
    '    "budgetTotal":          number (VND) or null,\n'
    '    "groupType":            "solo"|"couple"|"family"|"friends"|"business" or null,\n'
    '    "mobilityRestrictions": string[],\n'
    '    "dietaryPreferences":   string[],\n'
    '    "pace":                 float 0.0-1.0 or null,\n'
    '    "originalPrompt":       string\n'
    '  },\n'
    '  "missingSlots": string[],\n'
    '  "confidence":   float 0.0-1.0\n'
    '}\n\n'
    'Extraction rules:\n'
    '- startDate: ALWAYS output in YYYY-MM-DD format.\n'
    '    For relative dates, use the provided "today" context.\n'
    '    Example: If today is 2024-04-28, "cuoi tuan nay" -> "2024-05-04".\n'
    '    For explicit DD/MM/YYYY dates, convert them: "25/05/2026" -> "2026-05-25".\n'
    '    If no date mentioned, return null.\n'
    '- preferredTagNames: Infer category tags from ANY descriptive phrase, not just explicit keywords.\n'
    '    Valid tags: beach, mountain, nature, culture, food, spiritual, shopping, entertainment, sport, landmark.\n'
    '    Examples of inference:\n'
    '      "gio bien" (sea breeze), "bien ca", "bai bien" -> ["beach"]\n'
    '      "nui cao", "rung", "thac nuoc" -> ["mountain", "nature"]\n'
    '      "am thuc", "quan an", "ca phe" -> ["food"]\n'
    '      "chua", "den", "linh thieng" -> ["spiritual"]\n'
    '- experienceKeywords: Extract 10-20 descriptive phrases capturing desired EXPERIENCE and ATMOSPHERE.\n'
    '    Include sensory and spatial details:\n'
    '      "gio bien mat lanh", "canh hoang hon", "khong khi yen tinh", "gan duong lon", "view bien".\n'
    '    Keep longer descriptive phrases (e.g., "ngam hoang hon tren bien" not just "hoang hon").\n'
    '- vibe: Extract atmosphere (e.g., quiet, lively, romantic, modern, vintage, luxurious).\n'
    '- amenities: Extract requirements (e.g., wifi, parking, pool, gym, pet-friendly).\n'
    '- originalPrompt: Copy the entire user input exactly.\n'
    '- missingSlots: List slots that were NOT mentioned in the prompt.\n'
)
def _empty_slots():
    return {
        'destinationCity':      None,
        'durationDays':         None,
        'startDate':            None,
        'preferredTagNames':    [],
        'experienceKeywords':   [],
        'vibe':                 [],
        'amenities':            [],
        'budgetTotal':          None,
        'groupType':            None,
        'mobilityRestrictions': [],
        'dietaryPreferences':   [],
        'pace':                 None,
        'originalPrompt':       '',
    }
def _parse_date(sd):
    # Accept YYYY-MM-DD or DD/MM/YYYY, return YYYY-MM-DD string or None.
    if not isinstance(sd, str):
        return None
    # Already ISO
    if re.match(r'\d{4}-\d{2}-\d{2}', sd):
        return sd[:10]
    # Vietnamese DD/MM/YYYY
    m = re.match(r'(\d{1,2})/(\d{1,2})/(\d{4})', sd)
    if m:
        return f'{m.group(3)}-{m.group(2).zfill(2)}-{m.group(1).zfill(2)}'
    return None
def _normalise(raw):
    s = _empty_slots()
    GROUPS = {'solo', 'couple', 'family', 'friends', 'business'}
    s['destinationCity'] = raw.get('destinationCity') or None
    dur = raw.get('durationDays')
    s['durationDays'] = int(dur) if isinstance(dur, (int, float)) and dur > 0 else None
    s['startDate'] = _parse_date(raw.get('startDate'))
    s['preferredTagNames'] = [t.strip().lower() for t in raw.get('preferredTagNames', []) if isinstance(t, str)]
    exp = raw.get('experienceKeywords', [])
    if isinstance(exp, list):
        s['experienceKeywords'] = [e.strip() for e in exp if isinstance(e, str)][:20]
    s['vibe'] = [v.strip().lower() for v in raw.get('vibe', []) if isinstance(v, str)]
    s['amenities'] = [a.strip().lower() for a in raw.get('amenities', []) if isinstance(a, str)]
    budget = raw.get('budgetTotal')
    s['budgetTotal'] = float(budget) if isinstance(budget, (int, float)) and budget > 0 else None
    gt = raw.get('groupType')
    s['groupType'] = gt if gt in GROUPS else None
    s['mobilityRestrictions'] = [m for m in raw.get('mobilityRestrictions', []) if isinstance(m, str)]
    s['dietaryPreferences'] = [d for d in raw.get('dietaryPreferences', []) if isinstance(d, str)]
    pace = raw.get('pace')
    s['pace'] = round(float(pace), 2) if isinstance(pace, (int, float)) and 0.0 <= float(pace) <= 1.0 else None
    s['originalPrompt'] = raw.get('originalPrompt') or ''
    return s
def _extract_json(text):
    brace = text.find('{')
    if brace == -1: raise ValueError('No JSON found')
    depth, end = 0, brace
    for i, ch in enumerate(text[brace:], brace):
        if ch == '{': depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end = i
                break
    return text[brace:end+1]
def parse_prompt(prompt, today=None):
    today_ctx = f' (Today is {today})' if today else ''
    messages = [
        {'role': 'system', 'content': _SYSTEM + today_ctx},
        {'role': 'user',   'content': f'Parse this travel prompt: {prompt}'},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    raw_text = tokenizer.decode(out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    try:
        raw = json.loads(_extract_json(raw_text))
        slots = _normalise(raw.get('slots', {}))
        if not slots['originalPrompt']:
            slots['originalPrompt'] = prompt
        return {
            'slots': slots,
            'missingSlots': raw.get('missingSlots', []),
            'confidence': float(raw.get('confidence', 0.8)),
        }
    except Exception as e:
        print(f'Error parsing AI output: {e}\nRaw text: {raw_text}')
        raise e
print('NLU parser ready — DD/MM/YYYY date + semantic tag inference supported')

In [ ]:
# 4. Flask API - expose endpoint /api/nlu/parse cho backend
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'service': 'nlu-qwen'})


@app.route('/api/nlu/parse', methods=['POST'])
def nlu_parse():
    data = request.get_json(silent=True)
    if not data: return jsonify({'error': 'REQUEST_BODY_MISSING'}), 400
    prompt = data.get('prompt', '')
    today = data.get('today') # Context from backend
    
    if not isinstance(prompt, str) or not prompt.strip():
        return jsonify({'error': 'PROMPT_IS_EMPTY'}), 400
    try:
        return jsonify(parse_prompt(prompt.strip(), today))
    except json.JSONDecodeError as e:
        return jsonify({'error': f'MODEL_OUTPUT_NOT_JSON: {e}'}), 502
    except Exception as e:
        return jsonify({'error': str(e)}), 500

print('Flask app defined')

In [ ]:
# 5. Khoi dong pinggy tunnel + Flask server
# Pinggy tao HTTPS tunnel mien phi, khong can tai khoan
import subprocess, threading, re, time

public_url = None


def _start_pinggy():
    global public_url
    proc = subprocess.Popen(
        ['ssh', '-p', '443',
         '-R', f'0:localhost:{FLASK_PORT}',
         '-o', 'StrictHostKeyChecking=no',
         '-o', 'ServerAliveInterval=30',
         'a.pinggy.io'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print('[pinggy]', line.rstrip())
        m = re.search(r'https?://[\w.-]+\.pinggy\.link', line)
        if m:
            public_url = m.group()
            break


pinggy_thread = threading.Thread(target=_start_pinggy, daemon=True)
pinggy_thread.start()

# Doi pinggy lay URL (toi da 15 giay)
for _ in range(30):
    if public_url:
        break
    time.sleep(0.5)

if not public_url:
    print('WARN: Chua lay duoc public URL, kiem tra output pinggy o tren.')
else:
    print('=' * 60)
    print(f'NLU Server san sang')
    print(f'Public URL : {public_url}')
    print(f'Endpoint   : {public_url}/api/nlu/parse')
    print()
    print('Copy dong nay vao .env cua backend:')
    print(f'  COLAB_NLU_URL={public_url}')
    print('=' * 60)


def _run_flask():
    app.run(host='0.0.0.0', port=FLASK_PORT, use_reloader=False)


threading.Thread(target=_run_flask, daemon=True).start()

In [ ]:
# 6. Test thu cong (chay sau Cell 5)
import requests, json, time

time.sleep(1)
LOCAL = f'http://localhost:{FLASK_PORT}'

cases = [
    'Toi muon di Da Lat 3 ngay voi gia dinh, ngan sach 5 trieu, thich ca phe va thac nuoc',
    'Du lich Hoi An 5 ngay tu ngay 1/5, di cung nguoi yeu, budget 10 trieu, an chay, di thu thai',
    'Mot minh kham pha Ha Noi 2 ngay cuoi tuan, thich am thuc duong pho va van hoa',
    'Di Phu Quoc 7 ngay voi nhom ban, thich lan bien va chup anh, pace nhanh',
    # Cases test trich xuat experienceKeywords (cum mo ta trai nghiem)
    'Đi Đà Nẵng 3 ngày, ăn bánh mì chả cá và ngắm hoàng hôn ở biển',
    'Hội An 2 ngày, đi dạo phố cổ buổi tối, thích chụp ảnh đèn lồng',
]

for p in cases:
    label = (p[:70] + '...') if len(p) > 70 else p
    print(f'\nPrompt: {label}')
    r = requests.post(f'{LOCAL}/api/nlu/parse', json={'prompt': p}, timeout=60)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))
    print('-' * 60)